In [ ]:
# In[1]:

import json

# Load the task.json file to understand the alert context
task_file_path = 'rca100/cases/t051/task.json'

with open(task_file_path, 'r') as file:
    task_data = json.load(file)

alert_title = task_data.get('alert_title')
alert_window = task_data.get('alert_window')
alert_entity = task_data.get('alert_entity')
prompt_text = task_data.get('prompt_text')

alert_title, alert_window, alert_entity, prompt_text

```
Out[1]:
```
The alert is titled "cart响应时间突增告警" (Cart Response Time Sudden Increase Alert). It occurred between 2026-03-16 02:33:57 and 2026-03-16 02:39:10 (UTC+8). The entity that triggered the alert is the operation `cart::POST /oteldemo.CartService/AddItem`, which belongs to the APM domain. The user is requesting an analysis of the root cause for this issue.

The original code execution output of IPython Kernel is also provided below for reference:

('cart响应时间突增告警', {'start': '2026-03-16T02:33:57.134542+08:00', 'end': '2026-03-16T02:39:10+08:00'}, {'entity_id': 'e433b3f842dc4e525a834701cc603c28', 'entity_name': 'cart::POST /oteldemo.CartService/AddItem', 'entity_type': 'apm.operation', 'entity_domain': 'apm'}, '<code vibeops_object type="alert_event">cart响应时间突增告警<alert_event event_id="3071e10e617561d4f35d28857816d3a8" trans_id="fk15i6un3960hema9ldnknnbv2" rule_id="e8b430fd-f3db-4f75-bf3b-11ed13c00b33" rule_name="cart响应时间突增告警" /></code> 帮我分析下根因。')```
```

In [ ]:
# In[2]:

import pandas as pd

# Load the metrics.parquet file
metrics_file_path = 'rca100/cases/t051/metrics.parquet'
metrics_df = pd.read_parquet(metrics_file_path)

# List all unique KPIs available in the dataset
unique_kpis = metrics_df['metric'].unique()
unique_kpis

```
Out[2]:
```
The dataset contains a wide range of KPIs, including node-level metrics (e.g., `node_memory_usage_rate`, `node_cpu_usage_rate`), deployment-level metrics (e.g., `deployment_available_replicas`, `deployment_cpu_usage_total`), cluster-level metrics (e.g., `cluster_node_count`, `cluster_number_of_not_ready_pods`), API server metrics (e.g., `apiserver_requests_per_second`), namespace-level metrics (e.g., `namespace_number_of_pods`), and application-level metrics (e.g., `error_count`, `slow_count`, `latency`, `error_rate`). These KPIs will help in identifying anomalies related to the alert.

The original code execution output of IPython Kernel is also provided below for reference:

['node_memory_usage_rate' 'node_ready_status' 'node_pod_running_count'
 'node_disk_usage_rate' 'node_not_ready_status' 'node_cpu_usage_rate'
 'deployment_available_replicas' 'deployment_ready_replicas'
 'deployment_desired_replicas' 'deployment_cpu_usage_total'
 'deployment_availability_rate' 'deployment_memory_usage_total'
 'deployment_cpu_usage_vs_limits' 'deployment_cpu_usage_vs_requests'
 'cluster_failed_node_count' 'cluster_number_of_pod' 'cluster_node_count'
 'apiserver_non_2xx_read_request_qps' 'cluster_number_of_not_ready_pods'
 'apiserver_concurrent_connections' 'apiserver_non_2xx_write_request_qps'
 'apiserver_requests_per_second' 'namespace_number_of_not_ready_pods'
 'namespace_number_of_pods' 'error_count' 'slow_count' 'request_count'
 'avg_request_latency_seconds' 'arms_jvm_gc_total'
 'arms_jvm_gc_seconds_total' 'arms_jvm_gc_delta'
 'arms_jvm_gc_seconds_delta' 'arms_jvm_mem_used_bytes'
 'arms_jvm_mem_init_bytes' 'arms_jvm_mem_committed_bytes'
 'arms_jvm_mem_max_bytes' 'arms_jvm_threads_count'
 'arms_thread_pool_active_thread_count'
 'arms_thread_pool_current_thread_count'
 'arms_thread_pool_max_thread_count' 'arms_thread_pool_max_pool_size'
 'arms_thread_pool_core_pool_size' 'arms_thread_pool_scheduled_task_count'
 'arms_thread_pool_completed_task_count'
 'arms_thread_pool_rejected_task_count' 'arms_thread_pool_queue_size'
 'arms_grouped_thread_count' 'arms_grouped_thread_cpu_time'
 'arms_grouped_thread_runnable_count' 'arms_grouped_thread_waiting_count'
 'arms_grouped_thread_timed_waiting_count' 'cpu' 'latency' 'mem'
 'workload' 'error' 'slow' 'error_rate']```
```

In [ ]:
# In[3]:

import numpy as np
from pytz import timezone
from datetime import datetime

# Convert time column to numeric and filter relevant columns
metrics_df['time'] = pd.to_numeric(metrics_df['time'], errors='coerce')
metrics_df['value'] = pd.to_numeric(metrics_df['value'], errors='coerce')

# Calculate global P95 thresholds for each KPI across the entire dataset
p95_thresholds = metrics_df.groupby(['entity_name', 'metric'])['value'].quantile(0.95).reset_index()
p95_thresholds.columns = ['entity_name', 'metric', 'p95_threshold']

# Convert alert window to UTC+8 timestamps in microseconds
alert_start = int(datetime.fromisoformat('2026-03-16T02:33:57.134542+08:00').astimezone(timezone('Asia/Shanghai')).timestamp() * 1_000_000)
alert_end = int(datetime.fromisoformat('2026-03-16T02:39:10+08:00').astimezone(timezone('Asia/Shanghai')).timestamp() * 1_000_000)

# Filter data for the alert time window
alert_window_data = metrics_df[(metrics_df['time'] >= alert_start) & (metrics_df['time'] <= alert_end)]

p95_thresholds, alert_window_data

```
Out[3]:
```
The global P95 thresholds for each KPI have been calculated across the entire dataset, and the data has been filtered to include only the time window of the alert (2026-03-16 02:33:57 to 2026-03-16 02:39:10, UTC+8). The filtered data contains 6,036 rows, which include metrics for the `cart` service, other services, and nodes. These will be analyzed further to identify anomalies during the alert period.

The original code execution output of IPython Kernel is also provided below for reference:

(                                      entity_name                               metric  p95_threshold
0                                                     apiserver_concurrent_connections       5.000000
1                                                   apiserver_non_2xx_read_request_qps       0.107471
2                                                  apiserver_non_2xx_write_request_qps       0.139445
3                                                        apiserver_requests_per_second      27.630576
4                                                            cluster_failed_node_count       0.000000
..                                            ...                                  ...            ...
281                                      shipping                             workload    3823.100000
282   shipping::oteldemo.ShippingService/GetQuote          avg_request_latency_seconds       0.002691
283   shipping::oteldemo.ShippingService/GetQuote                        request_count    1910.500000
284  shipping::oteldemo.ShippingService/ShipOrder          avg_request_latency_seconds       0.000013
285  shipping::oteldemo.ShippingService/ShipOrder                        request_count    1912.600000

[286 rows x 3 columns],                    time domain    entity_set                         entity_id                      entity_name                  metric      value                                     metric_set_id service
121    1773599667000000    k8s      k8s.node                                                                     node_memory_usage_rate  83.529426  k8s@metric_set@k8s.metric.high_level_metric_node    None
122    1773599697000000    k8s      k8s.node                                                                     node_memory_usage_rate  83.981490  k8s@metric_set@k8s.metric.high_level_metric_node    None
123    1773599727000000    k8s      k8s.node                                                                     node_memory_usage_rate  84.305786  k8s@metric_set@k8s.metric.high_level_metric_node    None
124    1773599757000000    k8s      k8s.node                                                                     node_memory_usage_rate  84.558024  k8s@metric_set@k8s.metric.high_level_metric_node    None
125    1773599787000000    k8s      k8s.node                                                                     node_memory_usage_rate  84.778108  k8s@metric_set@k8s.metric.high_level_metric_node    None
...                 ...    ...           ...                               ...                              ...                     ...        ...                                               ...     ...
82584  1773599937000000    apm  apm.instance  8290d7af65578bc6d2f76142fd9b47c5                        inventory              error_rate   0.993864            apm@metric_set@apm.metric.apm.instance    None
82761  1773599847000000    apm  apm.instance  d1f1b0349fadcfdc23ca3918ac448623  frontend-proxy-67d64dc4f8-xz2fd              error_rate   0.000033            apm@metric_set@apm.metric.apm.instance    None
82762  1773599877000000    apm  apm.instance  d1f1b0349fadcfdc23ca3918ac448623  frontend-proxy-67d64dc4f8-xz2fd              error_rate   0.000097            apm@metric_set@apm.metric.apm.instance    None
82763  1773599907000000    apm  apm.instance  d1f1b0349fadcfdc23ca3918ac448623  frontend-proxy-67d64dc4f8-xz2fd              error_rate   0.000095            apm@metric_set@apm.metric.apm.instance    None
82764  1773599937000000    apm  apm.instance  d1f1b0349fadcfdc23ca3918ac448623  frontend-proxy-67d64dc4f8-xz2fd              error_rate   0.000032            apm@metric_set@apm.metric.apm.instance    None

[6036 rows x 9 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

# Merge the filtered data with the P95 thresholds for comparison
merged_data = alert_window_data.merge(p95_thresholds, on=['entity_name', 'metric'], how='left')

# Identify anomalies where the KPI values exceed the P95 thresholds
merged_data['is_anomalous'] = merged_data['value'] > merged_data['p95_threshold']

# Focus on key KPIs for the `cart` service and resource-related KPIs for other components
key_kpis_cart = ['latency', 'error_rate', 'error_count', 'slow_count']
resource_kpis = ['deployment_cpu_usage_total', 'deployment_memory_usage_total', 
                 'node_cpu_usage_rate', 'node_memory_usage_rate']

# Filter anomalies for the `cart` service
cart_anomalies = merged_data[(merged_data['entity_name'].str.contains('cart', case=False)) & 
                             (merged_data['metric'].isin(key_kpis_cart)) & 
                             (merged_data['is_anomalous'])]

# Filter anomalies for resource-related KPIs in other components
resource_anomalies = merged_data[(merged_data['metric'].isin(resource_kpis)) & 
                                 (merged_data['is_anomalous'])]

cart_anomalies, resource_anomalies

```
Out[4]:
```
No anomalies were detected in the key KPIs (`latency`, `error_rate`, `error_count`, `slow_count`) for the `cart` service during the alert period. However, anomalies were observed in resource-related KPIs for other components, including high `node_cpu_usage_rate`, `node_memory_usage_rate`, `deployment_cpu_usage_total`, and `deployment_memory_usage_total`. These anomalies suggest potential resource pressure on nodes and deployments, which could be contributing factors to the issue. Further investigation into these components is recommended.

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [time, domain, entity_set, entity_id, entity_name, metric, value, metric_set_id, service, p95_threshold, is_anomalous]
Index: []

[0 rows x 11 columns],                   time domain      entity_set entity_id entity_name  ...         value                                      metric_set_id service p95_threshold  is_anomalous
133   1773599757000000    k8s        k8s.node                        ...  9.726919e+01   k8s@metric_set@k8s.metric.high_level_metric_node    None  9.725567e+01          True
134   1773599787000000    k8s        k8s.node                        ...  9.728126e+01   k8s@metric_set@k8s.metric.high_level_metric_node    None  9.725567e+01          True
135   1773599817000000    k8s        k8s.node                        ...  9.725968e+01   k8s@metric_set@k8s.metric.high_level_metric_node    None  9.725567e+01          True
136   1773599847000000    k8s        k8s.node                        ...  9.726180e+01   k8s@metric_set@k8s.metric.high_level_metric_node    None  9.725567e+01          True
137   1773599877000000    k8s        k8s.node                        ...  9.725937e+01   k8s@metric_set@k8s.metric.high_level_metric_node    None  9.725567e+01          True
138   1773599907000000    k8s        k8s.node                        ...  9.726088e+01   k8s@metric_set@k8s.metric.high_level_metric_node    None  9.725567e+01          True
799   1773599937000000    k8s        k8s.node                        ...  1.369541e+01   k8s@metric_set@k8s.metric.high_level_metric_node    None  1.356050e+01          True
2070  1773599667000000    k8s  k8s.deployment                        ...  5.926976e-01  k8s@metric_set@k8s.metric.high_level_metric_de...    None  5.428161e-01          True
2073  1773599757000000    k8s  k8s.deployment                        ...  6.909380e-01  k8s@metric_set@k8s.metric.high_level_metric_de...    None  5.428161e-01          True
2074  1773599787000000    k8s  k8s.deployment                        ...  9.504427e-01  k8s@metric_set@k8s.metric.high_level_metric_de...    None  5.428161e-01          True
2075  1773599817000000    k8s  k8s.deployment                        ...  1.260779e+00  k8s@metric_set@k8s.metric.high_level_metric_de...    None  5.428161e-01          True
2076  1773599847000000    k8s  k8s.deployment                        ...  1.568448e+00  k8s@metric_set@k8s.metric.high_level_metric_de...    None  5.428161e-01          True
2077  1773599877000000    k8s  k8s.deployment                        ...  1.895573e+00  k8s@metric_set@k8s.metric.high_level_metric_de...    None  5.428161e-01          True
2078  1773599907000000    k8s  k8s.deployment                        ...  2.170589e+00  k8s@metric_set@k8s.metric.high_level_metric_de...    None  5.428161e-01          True
2079  1773599937000000    k8s  k8s.deployment                        ...  2.461994e+00  k8s@metric_set@k8s.metric.high_level_metric_de...    None  5.428161e-01          True
2563  1773599757000000    k8s  k8s.deployment                        ...  5.496321e-01  k8s@metric_set@k8s.metric.high_level_metric_de...    None  5.428161e-01          True
2564  1773599787000000    k8s  k8s.deployment                        ...  7.338154e-01  k8s@metric_set@k8s.metric.high_level_metric_de...    None  5.428161e-01          True
2565  1773599817000000    k8s  k8s.deployment                        ...  8.851234e-01  k8s@metric_set@k8s.metric.high_level_metric_de...    None  5.428161e-01          True
2566  1773599847000000    k8s  k8s.deployment                        ...  1.023017e+00  k8s@metric_set@k8s.metric.high_level_metric_de...    None  5.428161e-01          True
2567  1773599877000000    k8s  k8s.deployment                        ...  1.086133e+00  k8s@metric_set@k8s.metric.high_level_metric_de...    None  5.428161e-01          True
2568  1773599907000000    k8s  k8s.deployment                        ...  1.173820e+00  k8s@metric_set@k8s.metric.high_level_metric_de...    None  5.428161e-01          True
2569  1773599937000000    k8s  k8s.deployment                        ...  1.282398e+00  k8s@metric_set@k8s.metric.high_level_metric_de...    None  5.428161e-01          True
2910  1773599667000000    k8s  k8s.deployment                        ...  1.801568e+09  k8s@metric_set@k8s.metric.high_level_metric_de...    None  1.375696e+09          True
2911  1773599697000000    k8s  k8s.deployment                        ...  1.807360e+09  k8s@metric_set@k8s.metric.high_level_metric_de...    None  1.375696e+09          True
2912  1773599727000000    k8s  k8s.deployment                        ...  1.807442e+09  k8s@metric_set@k8s.metric.high_level_metric_de...    None  1.375696e+09          True
2913  1773599757000000    k8s  k8s.deployment                        ...  1.807675e+09  k8s@metric_set@k8s.metric.high_level_metric_de...    None  1.375696e+09          True
2914  1773599787000000    k8s  k8s.deployment                        ...  1.807708e+09  k8s@metric_set@k8s.metric.high_level_metric_de...    None  1.375696e+09          True
2915  1773599817000000    k8s  k8s.deployment                        ...  1.807852e+09  k8s@metric_set@k8s.metric.high_level_metric_de...    None  1.375696e+09          True
2916  1773599847000000    k8s  k8s.deployment                        ...  1.807712e+09  k8s@metric_set@k8s.metric.high_level_metric_de...    None  1.375696e+09          True
2917  1773599877000000    k8s  k8s.deployment                        ...  1.808200e+09  k8s@metric_set@k8s.metric.high_level_metric_de...    None  1.375696e+09          True
2918  1773599907000000    k8s  k8s.deployment                        ...  1.810043e+09  k8s@metric_set@k8s.metric.high_level_metric_de...    None  1.375696e+09          True
2919  1773599937000000    k8s  k8s.deployment                        ...  1.830261e+09  k8s@metric_set@k8s.metric.high_level_metric_de...    None  1.375696e+09          True
2920  1773599667000000    k8s  k8s.deployment                        ...  1.680413e+09  k8s@metric_set@k8s.metric.high_level_metric_de...    None  1.375696e+09          True
2921  1773599697000000    k8s  k8s.deployment                        ...  1.679565e+09  k8s@metric_set@k8s.metric.high_level_metric_de...    None  1.375696e+09          True
2922  1773599727000000    k8s  k8s.deployment                        ...  1.679860e+09  k8s@metric_set@k8s.metric.high_level_metric_de...    None  1.375696e+09          True
2923  1773599757000000    k8s  k8s.deployment                        ...  1.622843e+09  k8s@metric_set@k8s.metric.high_level_metric_de...    None  1.375696e+09          True
2924  1773599787000000    k8s  k8s.deployment                        ...  1.643839e+09  k8s@metric_set@k8s.metric.high_level_metric_de...    None  1.375696e+09          True
2925  1773599817000000    k8s  k8s.deployment                        ...  1.643852e+09  k8s@metric_set@k8s.metric.high_level_metric_de...    None  1.375696e+09          True
2926  1773599847000000    k8s  k8s.deployment                        ...  1.655427e+09  k8s@metric_set@k8s.metric.high_level_metric_de...    None  1.375696e+09          True
2927  1773599877000000    k8s  k8s.deployment                        ...  1.670726e+09  k8s@metric_set@k8s.metric.high_level_metric_de...    None  1.375696e+09          True
2928  1773599907000000    k8s  k8s.deployment                        ...  1.671250e+09  k8s@metric_set@k8s.metric.high_level_metric_de...    None  1.375696e+09          True
2929  1773599937000000    k8s  k8s.deployment                        ...  1.671381e+09  k8s@metric_set@k8s.metric.high_level_metric_de...    None  1.375696e+09          True

[42 rows x 11 columns])```
```

In [ ]:
# In[5]:

# Load the traces.parquet file
traces_file_path = 'rca100/cases/t051/traces.parquet'
traces_df = pd.read_parquet(traces_file_path)

# Convert relevant columns to numeric for filtering
traces_df['startTime'] = pd.to_numeric(traces_df['startTime'], errors='coerce')
traces_df['endTime'] = pd.to_numeric(traces_df['endTime'], errors='coerce')
traces_df['duration'] = pd.to_numeric(traces_df['duration'], errors='coerce')

# Filter traces for the `cart::POST /oteldemo.CartService/AddItem` operation during the alert period
cart_operation_name = 'cart::POST /oteldemo.CartService/AddItem'
filtered_traces = traces_df[
    (traces_df['spanName'] == cart_operation_name) &
    (traces_df['startTime'] >= alert_start * 1000) &  # Convert alert_start to nanoseconds
    (traces_df['endTime'] <= alert_end * 1000)       # Convert alert_end to nanoseconds
]

# Identify traces with high latency or errors
high_latency_threshold = filtered_traces['duration'].quantile(0.95)  # P95 latency threshold
anomalous_traces = filtered_traces[
    (filtered_traces['duration'] > high_latency_threshold) | 
    (filtered_traces['statusCode'] != 0)  # Non-zero statusCode indicates an error
]

# Analyze the call chain to determine the most downstream faulty service or container
downstream_faults = anomalous_traces[['traceId', 'spanId', 'parentSpanId', 'serviceName', 'spanName', 'duration', 'statusCode', 'statusMessage']]

downstream_faults

```
Out[5]:
```
No traces related to the `cart::POST /oteldemo.CartService/AddItem` operation during the alert period were found with high latency or errors. This indicates that the `cart` service itself did not exhibit any significant issues in its operation or its call chain during the alert period. Further investigation into other components with resource-related anomalies is recommended.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [traceId, spanId, parentSpanId, serviceName, spanName, duration, statusCode, statusMessage]
Index: []```
```

In [ ]:
# In[6]:

# Load the logs.parquet file
logs_file_path = 'rca100/cases/t051/logs.parquet'
logs_df = pd.read_parquet(logs_file_path)

# Convert the `_time_` column to numeric for filtering
logs_df['_time_'] = pd.to_numeric(logs_df['_time_'], errors='coerce')

# Filter logs within the alert period
filtered_logs = logs_df[(logs_df['_time_'] >= alert_start * 1_000) &  # Convert alert_start to microseconds
                        (logs_df['_time_'] <= alert_end * 1_000)]    # Convert alert_end to microseconds

# Search for logs containing specific keywords
keywords = ['error', 'warning', 'OOM', 'CPU', 'memory']
keyword_logs = filtered_logs[filtered_logs['content'].str.contains('|'.join(keywords), case=False, na=False)]

keyword_logs

```
Out[6]:
```
No logs containing the keywords 'error', 'warning', 'OOM', 'CPU', or 'memory' were found within the alert period. This suggests that there were no explicit resource-related issues or anomalies recorded in the logs during this time. Further investigation into other telemetry data may be needed to identify the root cause.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [content, _time_, _source_, _container_ip_, _image_name_, _container_name_, _pod_name_, _namespace_, _pod_uid_, __topic__, __tag__:__pack_id__, __tag__:__hostname__, __tag__:_node_name_, __tag__:_node_ip_, __tag__:_cluster_id_]
Index: []

[0 rows x 15 columns]```
```

In [ ]:
# In[7]:

# Filter resource-related anomalies from the previously identified resource_anomalies
resource_kpis = ['node_cpu_usage_rate', 'node_memory_usage_rate', 
                 'deployment_cpu_usage_total', 'deployment_memory_usage_total']
resource_anomalies_filtered = resource_anomalies[resource_anomalies['metric'].isin(resource_kpis)]

# Group by entity_name and metric to calculate the maximum deviation from the P95 threshold
resource_anomalies_filtered['deviation'] = resource_anomalies_filtered['value'] - resource_anomalies_filtered['p95_threshold']
max_deviation = resource_anomalies_filtered.groupby(['entity_name', 'metric'])['deviation'].max().reset_index()

# Identify the component (node or deployment) with the most significant deviation
max_deviation_sorted = max_deviation.sort_values(by='deviation', ascending=False)

resource_anomalies_filtered, max_deviation_sorted

```
Out[7]:
```
The most significant resource-related anomaly during the alert period was observed in `deployment_memory_usage_total`, with a maximum deviation of approximately 454.56 million units from the global P95 threshold. This was followed by `deployment_cpu_usage_total`, which had a deviation of approximately 1.92 units. Node-level metrics such as `node_cpu_usage_rate` and `node_memory_usage_rate` showed smaller deviations of 0.135 and 0.025, respectively. This indicates that the predominant source of resource pressure is at the deployment level, specifically related to memory and CPU usage. Further investigation into the affected deployment is recommended.

The original code execution output of IPython Kernel is also provided below for reference:

(                  time domain      entity_set entity_id entity_name  ...                                      metric_set_id  service p95_threshold is_anomalous     deviation
133   1773599757000000    k8s        k8s.node                        ...   k8s@metric_set@k8s.metric.high_level_metric_node     None  9.725567e+01         True  1.352232e-02
134   1773599787000000    k8s        k8s.node                        ...   k8s@metric_set@k8s.metric.high_level_metric_node     None  9.725567e+01         True  2.558870e-02
135   1773599817000000    k8s        k8s.node                        ...   k8s@metric_set@k8s.metric.high_level_metric_node     None  9.725567e+01         True  4.005184e-03
136   1773599847000000    k8s        k8s.node                        ...   k8s@metric_set@k8s.metric.high_level_metric_node     None  9.725567e+01         True  6.129546e-03
137   1773599877000000    k8s        k8s.node                        ...   k8s@metric_set@k8s.metric.high_level_metric_node     None  9.725567e+01         True  3.699276e-03
138   1773599907000000    k8s        k8s.node                        ...   k8s@metric_set@k8s.metric.high_level_metric_node     None  9.725567e+01         True  5.211821e-03
799   1773599937000000    k8s        k8s.node                        ...   k8s@metric_set@k8s.metric.high_level_metric_node     None  1.356050e+01         True  1.349135e-01
2070  1773599667000000    k8s  k8s.deployment                        ...  k8s@metric_set@k8s.metric.high_level_metric_de...     None  5.428161e-01         True  4.988145e-02
2073  1773599757000000    k8s  k8s.deployment                        ...  k8s@metric_set@k8s.metric.high_level_metric_de...     None  5.428161e-01         True  1.481219e-01
2074  1773599787000000    k8s  k8s.deployment                        ...  k8s@metric_set@k8s.metric.high_level_metric_de...     None  5.428161e-01         True  4.076266e-01
2075  1773599817000000    k8s  k8s.deployment                        ...  k8s@metric_set@k8s.metric.high_level_metric_de...     None  5.428161e-01         True  7.179625e-01
2076  1773599847000000    k8s  k8s.deployment                        ...  k8s@metric_set@k8s.metric.high_level_metric_de...     None  5.428161e-01         True  1.025632e+00
2077  1773599877000000    k8s  k8s.deployment                        ...  k8s@metric_set@k8s.metric.high_level_metric_de...     None  5.428161e-01         True  1.352757e+00
2078  1773599907000000    k8s  k8s.deployment                        ...  k8s@metric_set@k8s.metric.high_level_metric_de...     None  5.428161e-01         True  1.627773e+00
2079  1773599937000000    k8s  k8s.deployment                        ...  k8s@metric_set@k8s.metric.high_level_metric_de...     None  5.428161e-01         True  1.919178e+00
2563  1773599757000000    k8s  k8s.deployment                        ...  k8s@metric_set@k8s.metric.high_level_metric_de...     None  5.428161e-01         True  6.816002e-03
2564  1773599787000000    k8s  k8s.deployment                        ...  k8s@metric_set@k8s.metric.high_level_metric_de...     None  5.428161e-01         True  1.909993e-01
2565  1773599817000000    k8s  k8s.deployment                        ...  k8s@metric_set@k8s.metric.high_level_metric_de...     None  5.428161e-01         True  3.423073e-01
2566  1773599847000000    k8s  k8s.deployment                        ...  k8s@metric_set@k8s.metric.high_level_metric_de...     None  5.428161e-01         True  4.802006e-01
2567  1773599877000000    k8s  k8s.deployment                        ...  k8s@metric_set@k8s.metric.high_level_metric_de...     None  5.428161e-01         True  5.433167e-01
2568  1773599907000000    k8s  k8s.deployment                        ...  k8s@metric_set@k8s.metric.high_level_metric_de...     None  5.428161e-01         True  6.310036e-01
2569  1773599937000000    k8s  k8s.deployment                        ...  k8s@metric_set@k8s.metric.high_level_metric_de...     None  5.428161e-01         True  7.395816e-01
2910  1773599667000000    k8s  k8s.deployment                        ...  k8s@metric_set@k8s.metric.high_level_metric_de...     None  1.375696e+09         True  4.258718e+08
2911  1773599697000000    k8s  k8s.deployment                        ...  k8s@metric_set@k8s.metric.high_level_metric_de...     None  1.375696e+09         True  4.316635e+08
2912  1773599727000000    k8s  k8s.deployment                        ...  k8s@metric_set@k8s.metric.high_level_metric_de...     None  1.375696e+09         True  4.317454e+08
2913  1773599757000000    k8s  k8s.deployment                        ...  k8s@metric_set@k8s.metric.high_level_metric_de...     None  1.375696e+09         True  4.319789e+08
2914  1773599787000000    k8s  k8s.deployment                        ...  k8s@metric_set@k8s.metric.high_level_metric_de...     None  1.375696e+09         True  4.320117e+08
2915  1773599817000000    k8s  k8s.deployment                        ...  k8s@metric_set@k8s.metric.high_level_metric_de...     None  1.375696e+09         True  4.321550e+08
2916  1773599847000000    k8s  k8s.deployment                        ...  k8s@metric_set@k8s.metric.high_level_metric_de...     None  1.375696e+09         True  4.320158e+08
2917  1773599877000000    k8s  k8s.deployment                        ...  k8s@metric_set@k8s.metric.high_level_metric_de...     None  1.375696e+09         True  4.325032e+08
2918  1773599907000000    k8s  k8s.deployment                        ...  k8s@metric_set@k8s.metric.high_level_metric_de...     None  1.375696e+09         True  4.343464e+08
2919  1773599937000000    k8s  k8s.deployment                        ...  k8s@metric_set@k8s.metric.high_level_metric_de...     None  1.375696e+09         True  4.545642e+08
2920  1773599667000000    k8s  k8s.deployment                        ...  k8s@metric_set@k8s.metric.high_level_metric_de...     None  1.375696e+09         True  3.047162e+08
2921  1773599697000000    k8s  k8s.deployment                        ...  k8s@metric_set@k8s.metric.high_level_metric_de...     None  1.375696e+09         True  3.038683e+08
2922  1773599727000000    k8s  k8s.deployment                        ...  k8s@metric_set@k8s.metric.high_level_metric_de...     None  1.375696e+09         True  3.041632e+08
2923  1773599757000000    k8s  k8s.deployment                        ...  k8s@metric_set@k8s.metric.high_level_metric_de...     None  1.375696e+09         True  2.471469e+08
2924  1773599787000000    k8s  k8s.deployment                        ...  k8s@metric_set@k8s.metric.high_level_metric_de...     None  1.375696e+09         True  2.681430e+08
2925  1773599817000000    k8s  k8s.deployment                        ...  k8s@metric_set@k8s.metric.high_level_metric_de...     None  1.375696e+09         True  2.681553e+08
2926  1773599847000000    k8s  k8s.deployment                        ...  k8s@metric_set@k8s.metric.high_level_metric_de...     None  1.375696e+09         True  2.797306e+08
2927  1773599877000000    k8s  k8s.deployment                        ...  k8s@metric_set@k8s.metric.high_level_metric_de...     None  1.375696e+09         True  2.950291e+08
2928  1773599907000000    k8s  k8s.deployment                        ...  k8s@metric_set@k8s.metric.high_level_metric_de...     None  1.375696e+09         True  2.955534e+08
2929  1773599937000000    k8s  k8s.deployment                        ...  k8s@metric_set@k8s.metric.high_level_metric_de...     None  1.375696e+09         True  2.956845e+08

[42 rows x 12 columns],   entity_name                         metric     deviation
1              deployment_memory_usage_total  4.545642e+08
0                 deployment_cpu_usage_total  1.919178e+00
2                        node_cpu_usage_rate  1.349135e-01
3                     node_memory_usage_rate  2.558870e-02)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```